# Mini GPT → Mini LLM
### Smart Content Generation System using Decoder-Only Transformer
Problem Statement

A content-writing company wants an AI assistant capable of generating text suggestions while users write articles. Build a Mini GPT-based LLM that learns language patterns and generates text one token at a time.

# Task 1: Prepare Dataset

Use a text dataset such as:

1. TinyStories
2. WikiText-2
3. Custom articles dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hgultekin/bbcnewsarchive")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'bbcnewsarchive' dataset.
Path to dataset files: /kaggle/input/bbcnewsarchive


In [2]:
import pandas as pd
import os

files = os.listdir(path)
print(files)

['bbc-news-data.csv']


In [3]:
csv_file = os.path.join(path, files[0])

df = pd.read_csv(csv_file, sep='\t')

df.head()

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [4]:
print(df.shape)

print(df.columns)

print(df.info())

(2225, 4)
Index(['category', 'filename', 'title', 'content'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  2225 non-null   object
 1   filename  2225 non-null   object
 2   title     2225 non-null   object
 3   content   2225 non-null   object
dtypes: object(4)
memory usage: 69.7+ KB
None


In [5]:
texts = df['content'].astype(str).tolist()

In [6]:
print(df.columns)

Index(['category', 'filename', 'title', 'content'], dtype='object')


In [7]:
from collections import Counter

counter = Counter()

for text in texts:
    counter.update(text.split())

In [8]:
vocab = ["<PAD>", "<UNK>"] + list(counter.keys())

word2idx = {
    word: idx
    for idx, word in enumerate(vocab)
}

idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 64780


In [9]:
def tokenize(text):

    return [
        word2idx.get(word, word2idx["<UNK>"])
        for word in text.split()
    ]

In [10]:
max_len = 64

all_sequences = []

for text in texts:

    tokens = tokenize(text)

    if len(tokens) > max_len:

        tokens = tokens[:max_len]

    if len(tokens) > 2:

        all_sequences.append(tokens)

print(len(all_sequences))

2225


In [11]:
import torch

def pad_sequence(seq):

    if len(seq) < max_len:

        seq += [0] * (max_len - len(seq))

    return seq[:max_len]

In [12]:
from torch.utils.data import Dataset

class NewsDataset(Dataset):

    def __init__(self, sequences):

        self.sequences = sequences

    def __len__(self):

        return len(self.sequences)

    def __getitem__(self, idx):

        seq = pad_sequence(
            self.sequences[idx]
        )

        seq = torch.tensor(seq)

        return (
            seq[:-1],
            seq[1:]
        )

In [13]:
from torch.utils.data import DataLoader

dataset = NewsDataset(
    all_sequences
)

loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True
)

### Positional Encoding

In [14]:
import torch.nn as nn

class PositionalEmbedding(nn.Module):

    def __init__(
        self,
        max_len,
        d_model
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            max_len,
            d_model
        )

    def forward(self, x):

        positions = torch.arange(
            x.size(1),
            device=x.device
        ).unsqueeze(0)

        return self.embedding(
            positions
        )

In [15]:
class MaskedAttention(nn.Module):

    def __init__(
        self,
        d_model,
        heads
    ):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            d_model,
            heads,
            batch_first=True
        )

    def forward(self, x):

        seq_len = x.size(1)

        mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len
            ),
            diagonal=1
        ).bool().to(x.device)

        output, _ = self.attn(
            x,
            x,
            x,
            attn_mask=mask
        )

        return output

### GPT Decoder block

In [16]:
class GPTBlock(nn.Module):

    def __init__(
        self,
        d_model
    ):
        super().__init__()

        self.attn = MaskedAttention(
            d_model,
            4
        )

        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                d_model,
                4*d_model
            ),
            nn.ReLU(),
            nn.Linear(
                4*d_model,
                d_model
            )
        )

        self.norm2 = nn.LayerNorm(
            d_model
        )

    def forward(self, x):

        x = self.norm1(
            x + self.attn(x)
        )

        x = self.norm2(
            x + self.ffn(x)
        )

        return x

In [17]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_len=64,
        d_model=128,
        layers=4
    ):
        super().__init__()

        self.token_embed = nn.Embedding(
            vocab_size,
            d_model
        )

        self.pos_embed = PositionalEmbedding(
            max_len,
            d_model
        )

        self.blocks = nn.ModuleList([
            GPTBlock(d_model)
            for _ in range(layers)
        ])

        self.norm = nn.LayerNorm(
            d_model
        )

        self.fc = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, x):

        x = (
            self.token_embed(x)
            + self.pos_embed(x)
        )

        for block in self.blocks:

            x = block(x)

        x = self.norm(x)

        return self.fc(x)

In [18]:
vocab_size = len(vocab)

model = MiniGPT(
    vocab_size
)

In [19]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(device)

MiniGPT(
  (token_embed): Embedding(64780, 128)
  (pos_embed): PositionalEmbedding(
    (embedding): Embedding(64, 128)
  )
  (blocks): ModuleList(
    (0-3): 4 x GPTBlock(
      (attn): MaskedAttention(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): ReLU()
        (2): Linear(in_features=512, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (fc): Linear(in_features=128, out_features=64780, bias=True)
)

In [20]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0003
)

In [21]:
criterion = nn.CrossEntropyLoss()

### Train Model

In [32]:
epochs = 20

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        output = model(x)

        loss = criterion(
            output.reshape(
                -1,
                vocab_size
            ),
            y.reshape(-1)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: {total_loss/len(loader):.4f}"
    )

Epoch 1: 6.7649
Epoch 2: 6.4974
Epoch 3: 6.2228
Epoch 4: 5.9501
Epoch 5: 5.6663
Epoch 6: 5.3814
Epoch 7: 5.0835
Epoch 8: 4.7926
Epoch 9: 4.4972
Epoch 10: 4.2099
Epoch 11: 3.9207
Epoch 12: 3.6497
Epoch 13: 3.3751
Epoch 14: 3.1199
Epoch 15: 2.8637
Epoch 16: 2.6261
Epoch 17: 2.4140
Epoch 18: 2.2070
Epoch 19: 2.0119
Epoch 20: 1.8359


### Text Generation

In [33]:
def generate_text(
    prompt,
    max_tokens=50
):

    model.eval()

    tokens = tokenize(prompt)

    tokens = torch.tensor(
        [tokens]
    ).to(device)

    with torch.no_grad():

        for _ in range(max_tokens):

            output = model(tokens)

            next_token = torch.argmax(
                output[:, -1, :],
                dim=-1
            )

            tokens = torch.cat(
                [
                    tokens,
                    next_token.unsqueeze(1)
                ],
                dim=1
            )

    words = []

    for idx in tokens[0]:

        words.append(
            idx2word.get(
                idx.item(),
                "<UNK>"
            )
        )

    return " ".join(words)

In [34]:
print(
    generate_text(
        "Artificial Intelligence"
    )
)

<UNK> Intelligence Minister have been honoured for three quarter. show on Friday and have been knighted Awards. by the firm on Thursday. The soap figures, with France's Arcleor in limbo festival with a best-selling novel "I have been charged included in Westminster, by the last July. However Nasdaq, best film and Irish


In [35]:
while True:

    prompt = input("Enter Prompt (type exit to stop): ")

    if prompt.lower() == "exit":
        break

    output = generate_text(prompt)

    print("\nGenerated Text:")
    print(output)
    print("-" * 50)

Enter Prompt (type exit to stop): Politics

Generated Text:
Politics MP believes the UK have been named the top of hit record in two games market. The 25-year-old Fry will be of 2004 for technology appeal in the British the Oscars. time on Tuesday, on Friday. The figures showed Association of 10 have confirmed she was the IAAF and global
--------------------------------------------------
Enter Prompt (type exit to stop): exit


In [26]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [27]:
tokens = tokenizer.encode(text)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3545 > 1024). Running this sequence through the model will result in indexing errors


In [28]:
while True:

    prompt = input("Enter Prompt (type exit to stop): ")

    if prompt.lower() == "exit":
        break

    output = generate_text(prompt)

    print("\nGenerated Text:")
    print(output)
    print("-" * 50)


Enter Prompt (type exit to stop): breaking news

Generated Text:
breaking news has been a new the US of the US of the US of the US The US The US The US The US The US is to be able to be a new for the US The US is to be a new US The US is to be been
--------------------------------------------------
Enter Prompt (type exit to stop): exit


In [29]:
df.columns


Index(['category', 'filename', 'title', 'content'], dtype='object')

In [31]:
df.head()

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [37]:

while True:

    prompt = input("Enter Prompt (type exit to stop): ")

    if prompt.lower() == "exit":
        break

    output = generate_text(prompt)

    print("\nGenerated Text:")
    print(output)
    print("-" * 50)

Enter Prompt (type exit to stop): British Airways

Generated Text:
British Airways has defended his second time the US stock market the three months to a poll important round one in Rome. to bolster the market in the New Australian Open to take place to fourth and Real Madrid But of 204.3m his other disasters. to overtake him, was looking to the
--------------------------------------------------
Enter Prompt (type exit to stop): exit
